In [1]:
import pandas as pd
import numpy as np
import re

def parse_mixed_date(value):
    text = str(value).strip()
    if re.match(r"^\d{4}-\d{2}-\d{2}$", text):
        return pd.to_datetime(text, format="%Y-%m-%d", errors="coerce")
    if re.match(r"^\d{2}/\d{2}/\d{4}$", text):
        return pd.to_datetime(text, format="%m/%d/%Y", errors="coerce")
    if re.match(r"^\d{2}-\d{2}-\d{4}$", text):
        return pd.to_datetime(text, format="%d-%m-%Y", errors="coerce")
    return pd.NaT


In [2]:
df = pd.read_csv("/content/expense_data_raw.csv")

print("Raw rows:", len(df))
print(df.isna().sum())

Raw rows: 104
date               0
user_id            4
user_name          0
category           0
amount            12
payment_method     0
dtype: int64


In [3]:
df["amount"] = df["amount"].astype(str).str.replace(r"[₹$,]", "", regex=True).str.replace("Rs.", "", regex=False)
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df["date"] = df["date"].apply(parse_mixed_date)
df["user_id"] = pd.to_numeric(df["user_id"], errors="coerce")

In [4]:
df = df.drop_duplicates()
df = df.dropna(subset=["user_id", "date"])
df["user_id"] = df["user_id"].astype(int)

In [5]:
df.loc[df["amount"] < 0, "amount"] = np.nan

category_medians = df.groupby("category")["amount"].transform("median")
df["amount"] = df["amount"].fillna(category_medians)

In [6]:
df = df.dropna(subset=["amount"])

print("\nCleaned rows:", len(df))

df["month"] = df["date"].dt.to_period("M")


Cleaned rows: 94


In [7]:
monthly_totals = df.groupby("month")["amount"].apply(lambda x: np.sum(x.to_numpy())).round(2)
monthly_averages = df.groupby("month")["amount"].apply(lambda x: np.mean(x.to_numpy())).round(2)

monthly_summary = pd.DataFrame({
    "total_amount": monthly_totals,
    "average_transaction": monthly_averages
})

print("\nMonthly totals and averages:")
print(monthly_summary)


Monthly totals and averages:
         total_amount  average_transaction
month                                     
2026-05      57818.75              1179.97
2026-06      52319.35              1162.65


In [8]:
category_breakdown = df.groupby(["month", "category"])["amount"].sum().unstack().fillna(0).round(2)

print("\nMonthly breakdown by category:")
print(category_breakdown)


Monthly breakdown by category:
category  Dining Out  Entertainment  Groceries  Healthcare  Shopping  \
month                                                                  
2026-05      9343.39        6777.65    6243.92     7091.15  13761.23   
2026-06      4203.26        3619.30   15634.45    11054.88   8771.25   

category  Transportation  Utilities  
month                                
2026-05          3437.36   11164.06  
2026-06          3498.78    5537.43  


In [9]:
user_monthly_summary = df.groupby(["month", "user_name"])["amount"].sum().unstack().fillna(0).round(2)

print("\nMonthly spend by user:")
print(user_monthly_summary)


Monthly spend by user:
user_name  Arjun Mehta  Karthik Iyer  Priya Nair  Rohan Verma  Sneha Reddy
month                                                                     
2026-05       13619.50       9537.33    10086.24     14437.11     10138.58
2026-06        8846.66      17540.08     6975.36     12293.07      6664.18


In [10]:
df.to_csv("/content/expense_data_cleaned.csv", index=False)
monthly_summary.to_csv("/content/monthly_summary.csv")
category_breakdown.to_csv("/content/monthly_category_breakdown.csv")
user_monthly_summary.to_csv("/content/monthly_user_breakdown.csv")
